Load necessary dependencies.

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy.odr import ODR, Model, Data

Load data.

In [2]:
beams_data = np.loadtxt('beams.csv', delimiter=',', skiprows=1)

X = beams_data[:, 0]  # Specific Gravity
y = beams_data[:, 1]  # Strength

Fit and run ordinary least squares model.

In [3]:
simple_model = LinearRegression().fit(X.reshape(-1, 1), y)
print(f"Prediction at Specific Gravity 0.590: {simple_model.predict([[0.590]])[0]}")

Prediction at Specific Gravity 0.590: 12.903712789598174


Using closed form to do ordinary least squares.

In [4]:
sxx = np.sum((X - np.mean(X))**2)
sxy = np.sum((X - np.mean(X)) * (y - np.mean(y)))
beta1 = sxy / sxx
beta0 = np.mean(y) - beta1 * np.mean(X)
print(f"Prediction at Specific Gravity 0.590: {beta1 * 0.590 + beta0}")

Prediction at Specific Gravity 0.590: 12.903712789598174


Fit and run an orthogonal least squares model (Orthogonal Distance Regression).

In [5]:
def linear_model(B, x):
    return B[0] * x + B[1]

model = Model(linear_model)
data = Data(X, y)

odr = ODR(data, model, beta0=[1.0, 0.0])
param = odr.run().beta
print(f"Prediction at Specific Gravity 0.590: {param[0] * 0.590 + param[1]}")

Prediction at Specific Gravity 0.590: 13.107093827127159


Using closed form to do orthogonal least squares.

In [6]:
sxx = np.sum((X - np.mean(X))**2)
sxy = np.sum((X - np.mean(X)) * (y - np.mean(y)))
syy = np.sum((y - np.mean(y))**2)

beta1 = ((syy - sxx) + np.sqrt((syy - sxx)**2 + 4 * sxy**2)) / (2 * sxy)
beta0 = np.mean(y) - beta1 * np.mean(X)

print(f"Prediction at Specific Gravity 0.590: {beta1 * 0.590 + beta0}")

Prediction at Specific Gravity 0.590: 13.107093620833423


Fit and run an Errors in Variables model.

In [15]:
eiv_data = Data(X, y, wd=1.0/0.01, we=1.0/1)
eiv_obj = ODR(eiv_data, model, beta0=[1.0, 0.0])
eiv_parameters = eiv_obj.run().beta

print(f"Prediction at Specific Gravity 0.590: {eiv_parameters[0] * 0.590 + eiv_parameters[1]}")

Prediction at Specific Gravity 0.590: 13.019673997367835


Using closed form to do error in variables.

In [ ]:
lam = 0.01
sxx = np.sum((X - np.mean(X))**2)
sxy = np.sqrt(lam) * np.sum((X - np.mean(X)) * (y - np.mean(y)))
syy = lam * np.sum((y - np.mean(y))**2)

beta1 = ((syy - sxx) + np.sqrt((syy - sxx)**2 + 4 * sxy**2)) / (2 * sxy)
beta0 = np.mean(np.sqrt(lam) * y) - beta1 * np.mean(X)

beta1 = beta1 / np.sqrt(lam)
beta0 = beta0 / np.sqrt(lam)

print(f"Prediction at Specific Gravity 0.590: {beta1 * 0.590 + beta0}")

Prediction at Specific Gravity 0.590: 119.90368259719115
